# Proyek Pengembangan Machine Learning Pipeline (Dicoding Submission 1)

- **Nama**: Sonny Ariady
- **Username Dicoding**: sonnyariady
- **Dataset**: Heart Disease Dataset (Binary Classification - Health)
- **Komponen TFX**: ExampleGen, StatisticGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Resolver, Evaluator, Pusher

Notebook ini mengimplementasikan Machine Learning Pipeline terautomasi secara end-to-end berbasis **TensorFlow Extended (TFX)** menggunakan `InteractiveContext`.

## 1. Import Library & Pengaturan Environment

In [ ]:
import os
import sys
import tensorflow as tf
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
import tensorflow_model_analysis as tfma

print(f"TensorFlow Version: {tf.__version__}")

## 2. Inisialisasi InteractiveContext & Directory Pipeline

In [ ]:
USERNAME = "sonnyariady"
PIPELINE_NAME = f"{USERNAME}-pipeline"
PIPELINE_ROOT = os.path.join(PIPELINE_NAME)
DATA_ROOT = "data"
SERVING_MODEL_DIR = os.path.join("serving_model", "heart-disease-model")

# Inisialisasi InteractiveContext dengan pipeline_root sesuai kriteria
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)
print(f"InteractiveContext initialized at: {PIPELINE_ROOT}")

## 3. Komponen 1: ExampleGen
`ExampleGen` membaca dataset CSV dari direktori `data/` dan membaginya menjadi data latih (train) dan evaluasi (eval) dalam format TFRecord.

In [ ]:
example_gen = CsvExampleGen(input_base=DATA_ROOT)
context.run(example_gen)

## 4. Komponen 2: StatisticGen
`StatisticGen` mengomputasi statistik deskriptif untuk data latih dan evaluasi.

In [ ]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)
context.show(statistics_gen.outputs['statistics'])

## 5. Komponen 3: SchemaGen
`SchemaGen` menginferensi skema data berdasarkan hasil statistik (tipe data, rentang nilai, ketersediaan fitur).

In [ ]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)
context.run(schema_gen)
context.show(schema_gen.outputs['schema'])

## 6. Komponen 4: ExampleValidator
`ExampleValidator` mendeteksi adanya anomali atau penyimpangan data terhadap skema yang telah dibuat.

In [ ]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
context.run(example_validator)
context.show(example_validator.outputs['anomalies'])

## 7. Komponen 5: Transform
`Transform` melakukan preprocessing fitur data (normalisasi fitur numerik dan pencocokan vocabulary fitur kategorikal) menggunakan modul `transform.py`.

In [ ]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file='transform.py'
)
context.run(transform)

## 8. Komponen 6: Tuner (Saran 1 ⭐)
Komponen `Tuner` menjalankan otomatisasi hyperparameter tuning menggunakan modul `tuner.py` dan KerasTuner.

In [ ]:
tuner = Tuner(
    module_file='tuner.py',
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=10)
)
context.run(tuner)

## 9. Komponen 7: Trainer
`Trainer` melatih model Deep Neural Network (DNN) menggunakan modul `trainer.py` dan hyperparameter terbaik dari `Tuner`.

In [ ]:
trainer = Trainer(
    module_file='trainer.py',
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=30),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=10)
)
context.run(trainer)

## 10. Komponen 8: Resolver
`Resolver` mengambil model *blessed* sebelumnya dari metadata untuk dijadikan baseline dalam evaluasi model candidate.

In [ ]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
)
context.run(model_resolver)

## 11. Komponen 9: Evaluator
`Evaluator` memvalidasi performa model candidate menggunakan TFMA (`tensorflow_model_analysis`) terhadap ambang batas metrik (Accuracy > 0.75) dan baseline model.

In [ ]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(label_key='target')
    ],
    slicing_specs=[
        tfma.SlicingSpec(),
        tfma.SlicingSpec(feature_keys=['sex'])
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.70}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -1e-10}
                        )
                    )
                ),
                tfma.MetricConfig(class_name='AUC'),
                tfma.MetricConfig(class_name='Precision'),
                tfma.MetricConfig(class_name='Recall')
            ]
        )
    ]
)

evaluator = Evaluator(
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config,
    examples=example_gen.outputs['examples']
)
context.run(evaluator)

## 12. Komponen 10: Pusher
Jika model candidate lolos evaluasi (*Blessed*), `Pusher` mengekspor model ke direktori serving (`serving_model/`).

In [ ]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)
context.run(pusher)
print("Pipeline execution complete! Serving model pushed to:", SERVING_MODEL_DIR)